# Tutorial 11: Lazy Entries with Constitutions

A LAILA entry can carry a **constitution** instead of a concrete payload. The constitution describes *how* to compute the payload, and the entry starts in `STAGED` state. Calling `laila.build(entry)` runs the recipe on a taskforce and flips the entry to `READY`.

LAILA ships two constitution flavours:

- **`SimpleConstitution`** — an ordered chain of single-callable code strings; primarily used internally as the inverse of a serializer pipeline.
- **`ComplexConstitution`** — a single Python source string defining `f(manifest) -> payload`, bound to a `Manifest` of input entries.

You will:

- Build a `STAGED` entry from a `ComplexConstitution`
- Demonstrate that `entry.data` raises `EntryNotBuiltError` until `build` runs
- See `await laila.build(entry)` from an async context
- Inspect how `SimpleConstitution` runs an inverse serialization chain

**Prerequisites:** `pip install laila-core`.

In [ ]:
import laila
from laila.entry import Entry, EntryState
from laila.entry.constitution import SimpleConstitution, ComplexConstitution
from laila.entry.exceptions import EntryNotBuiltError
from laila.policy.central.memory.schema.manifest import Manifest
from laila.macros.defaults import DefaultPool

laila.memory.extend(DefaultPool(), pool_nickname="constitutions")

## A complex constitution

The recipe is a Python source string that defines exactly one callable taking a `Manifest` and returning the payload. The manifest's leaves are themselves entries — at build time, LAILA resolves every leaf and hands the materialized manifest to your function.

In [ ]:
a = laila.constant(data=10)
b = laila.constant(data=32)
manifest = Manifest(data={"a": a, "b": b})
manifest.memorize().wait()

CONSTITUTION_SRC = (
    "def sum_entries(manifest):\n"
    "    d = manifest.realized\n"
    "    return d['a'].data + d['b'].data\n"
)

entry = Entry.variable(constitution=CONSTITUTION_SRC, manifest=manifest)
print("state before build:", entry.state)
print("constitution attached:", entry.constitution is not None)

## `entry.data` raises until built

Reading `.data` on a staged entry is an explicit error — there is no implicit lazy build, you must call `laila.build` yourself.

In [ ]:
try:
    entry.data
except EntryNotBuiltError as e:
    print("caught EntryNotBuiltError:", e)

## `laila.build` materializes the entry

`laila.build(entry)` returns a future. After it resolves, the entry's `state` is `READY`, the constitution is cleared, and `entry.data` is the computed payload.

In [ ]:
future = laila.build(entry)
future.wait()

print("state after build:", entry.state)
print("constitution cleared:", entry.constitution is None)
print("payload:", entry.data)

## Same future, async style

The future returned by `build` is awaitable. From an `async def` you can write `await laila.build(entry)` instead of `.wait()`.

In [ ]:
import asyncio

async def build_async():
    entry2 = Entry.variable(constitution=CONSTITUTION_SRC, manifest=manifest)
    await laila.build(entry2)
    return entry2.data

print("async build result:", asyncio.run(build_async()))

## SimpleConstitution — inverse serialization chains

A `SimpleConstitution` carries an ordered list of source strings. `build(payload_input)` threads the input through each callable left-to-right. This is the same machinery LAILA uses internally to invert a pool's transformation sequence on read.

In [ ]:
import base64, zlib, msgpack

decode_chain = SimpleConstitution(codes=[
    "def b64(x): import base64; return base64.b64decode(x)",
    "def unz(x): import zlib; return zlib.decompress(x)",
    "def unp(x): import msgpack; return msgpack.unpackb(x)",
])

original = {"hello": "world", "n": 42}
encoded = base64.b64encode(zlib.compress(msgpack.packb(original)))

print("encoded bytes (truncated):", encoded[:24], "...")
print("decoded via constitution:", decode_chain.build(payload_input=encoded))

## Persisting a constitution-driven entry

A `STAGED` entry cannot be memorized — build it first, then memorize the materialized result. After round-tripping, the recovered entry is a plain `READY` entry with the concrete payload.

In [ ]:
staged = Entry.variable(constitution=CONSTITUTION_SRC, manifest=manifest)
laila.build(staged).wait()
laila.memorize(staged, pool_nickname="constitutions").wait()

recovered = laila.remember(staged.global_id, pool_nickname="constitutions", persist=False).wait()
if isinstance(recovered, list):
    recovered = recovered[0]
print("recovered state:", recovered.state, "data:", recovered.data)

## When to reach for constitutions

| Pattern | Use a constitution? |
|---|---|
| Caching the output of an expensive function | Yes — the constitution makes the recipe portable; recompute or reload as needed. |
| Composing a derived dataset from inputs in different pools | Yes — the manifest names the inputs by gid, so the build runs anywhere they're reachable. |
| Plain data with no derivation | No — use `laila.constant` / `laila.variable` directly. |

## Summary

- `Entry.variable(constitution=src, manifest=m)` produces a `STAGED` entry.
- `laila.build(entry)` runs the recipe; both `.wait()` (sync) and `await` (async) work on the returned future.
- Once built, the constitution is cleared and the entry is a plain `READY` entry.
- `SimpleConstitution` is for ordered transform chains; `ComplexConstitution` is for arbitrary derivations driven by a manifest.